# matched v2.1.1 — kiểm chứng score cube, H1 Holm và H2 non-inferiority

Notebook này **không train mô hình**. Sau khi sáu notebook v2.1.1 chạy xong, attach cả sáu output rồi chạy notebook này.

Nó hard-code hash của sáu protocol/code, tính lại Macro-F1 từ từng `score_cube.npz`, kiểm tra test ID/label trùng nhau giữa RF+, LightGBM+ và Meta-IFD ở cả 20 split, sau đó mới thực hiện H2 non-inferiority và gated superiority. Master protocol SHA-256: `3c50b13557fc897d71787171b2a98be31b8784de4be6fbfa6932ab063ebd1145`.


In [ ]:

from pathlib import Path
import csv
import hashlib
import json

import numpy as np
from scipy.stats import t as student_t
from sklearn.metrics import f1_score

MASTER_PROTOCOL_SHA256 = "3c50b13557fc897d71787171b2a98be31b8784de4be6fbfa6932ab063ebd1145"
EXPECTED_SPLIT_SEEDS = [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
EXPECTED_HASHES = {'logistic_regression': {'protocol_sha256': 'df53fb0710ea96cd8d3df9e6f8d5a675ca74ca804daabce811901cad553a4651', 'code_sha256': '1dce6a3175686872af445287ce6d4c1c49f571611185045aa4c9e00541065156'}, 'random_forest': {'protocol_sha256': '9627e2dbd9fa911fe5bb72e0782f5a2e22a328bbc5b5bcfae95a48207863e29d', 'code_sha256': '5bb6c36f19dc62d76a4346c67e17025072f00d24af9085a694708eb4b21282f6'}, 'lightgbm': {'protocol_sha256': '08ed7a1b3001db6dabebb5b6d094f5c577245d77f7ab01e4df183c5e249a662f', 'code_sha256': '930ad5b2991d4e1903157dafeff1c1e8e050062f45d7b2327120ebaa2afc7cd6'}, 'gat': {'protocol_sha256': '6490d2c83e63fee05a4a619c2a3705973d4fdfab70f549e155cc5896ff1d542e', 'code_sha256': 'f946857d8e064e7d282ed95c628a325463ad29befdb0b4722bd5a84bee90fafb'}, 'graphsage': {'protocol_sha256': '79736bc8387da504acd46f32a146933a28e7a5afb1b0f6515e7d5f6170754baa', 'code_sha256': 'c8d359724ae0e4433e46dfc080088b130969699c1d22aab66314ed9f88a93290'}, 'metaifd': {'protocol_sha256': '6c4a8f4b70b8b11ee86016b5116b0a2a2a391ba60a705986cb5c9623774f41ca', 'code_sha256': 'a0901dba85138cfacf0d6fef20fdcb23152bc32b32bf219bb03286e7c419dcb7'}}
EXPECTED = {
    "logistic_regression": ("LR", "LR+"),
    "random_forest": ("RF", "RF+"),
    "lightgbm": ("LightGBM", "LightGBM+"),
    "gat": ("GAT-Norm", "GAT+"),
    "graphsage": ("GraphSAGE-Norm", "GraphSAGE+"),
    "metaifd": ("Meta-IFD", "Meta-IFD+"),
}
ALPHA = 0.05
H2_ONE_SIDED_ALPHA = 0.025
MINIMUM_PRACTICAL_DELTA = 0.005
NONINFERIORITY_MARGIN = -0.01
RATIO_TEST_TRAIN = 553 / 1657


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def holm_adjust(p_values):
    m = len(p_values)
    order = sorted(range(m), key=lambda i: p_values[i])
    adjusted = [None] * m
    running = 0.0
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p_values[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted


def mean_macro(score_matrix, labels):
    assert score_matrix.shape == (5, len(labels))
    return float(np.mean([
        f1_score(labels, scores >= 0.5, average="macro", zero_division=0)
        for scores in score_matrix
    ]))


def nb_stats(deltas):
    deltas = np.asarray(deltas, dtype=float)
    k = len(deltas)
    assert k == 20
    variance = float(deltas.var(ddof=1))
    se = float(np.sqrt((1 / k + RATIO_TEST_TRAIN) * variance))
    mean = float(deltas.mean())
    if se == 0:
        t_zero = float("inf") if mean > 0 else (float("-inf") if mean < 0 else 0.0)
        p_two = 0.0 if mean != 0 else 1.0
        ci = (mean, mean)
        p_ni = 0.0 if mean > NONINFERIORITY_MARGIN else 1.0
        p_sup = 0.0 if mean > 0 else 1.0
    else:
        crit = float(student_t.ppf(0.975, df=k - 1))
        ci = (mean - crit * se, mean + crit * se)
        t_zero = mean / se
        p_two = float(2 * student_t.sf(abs(t_zero), df=k - 1))
        p_ni = float(student_t.sf((mean - NONINFERIORITY_MARGIN) / se, df=k - 1))
        p_sup = float(student_t.sf(mean / se, df=k - 1))
    return {
        "mean_delta": mean,
        "sd_delta": float(np.sqrt(variance)),
        "se_nb": se,
        "t_vs_zero": t_zero,
        "df": k - 1,
        "ci95": [float(ci[0]), float(ci[1])],
        "p_two_sided_vs_zero": p_two,
        "p_one_sided_noninferiority_raw": p_ni,
        "p_one_sided_superiority_raw": p_sup,
        "ratio_test_train": RATIO_TEST_TRAIN,
    }


candidates = []
for path in Path("/kaggle/input").rglob("final_summary.json"):
    try:
        item = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if item.get("master_protocol_sha256") == MASTER_PROTOCOL_SHA256:
        candidates.append((path, item))

by_key = {}
for path, item in candidates:
    key = item.get("model_key")
    assert key in EXPECTED, (path, key)
    expected = EXPECTED_HASHES[key]
    assert item.get("protocol_sha256") == expected["protocol_sha256"], (path, "protocol_sha256")
    assert item.get("code_sha256") == expected["code_sha256"], (path, "code_sha256")
    assert item["protocol"].get("code_sha256") == expected["code_sha256"], (path, "embedded code_sha256")
    assert item.get("all_20_splits_completed") is True, path
    assert item.get("post_test_selection") is False, path
    assert item.get("protocol_changed_after_results") is False, path
    block = item["v2_20split"]
    protocol_seeds = item["protocol"].get("split_seeds")
    result_seeds = [int(row["split_key"].split("_")[-1]) for row in block["per_split"]]
    assert protocol_seeds == EXPECTED_SPLIT_SEEDS, (path, protocol_seeds)
    assert result_seeds == EXPECTED_SPLIT_SEEDS, (path, result_seeds)
    assert len(block["per_split"]) == 20
    assert key not in by_key, f"Duplicate pair: {key} from {path} and {by_key[key][0]}"
    by_key[key] = (path, item)

assert set(by_key) == set(EXPECTED), {
    "found": sorted(by_key),
    "missing": sorted(set(EXPECTED) - set(by_key)),
}

# Recompute every split from score_cube and verify hashes, test IDs and metrics.
verified = {key: {} for key in EXPECTED}
for key, (summary_path, item) in by_key.items():
    run_root = summary_path.parent
    expected_hash = EXPECTED_HASHES[key]
    final_rows = {
        int(row["split_key"].split("_")[-1]): row
        for row in item["v2_20split"]["per_split"]
    }
    for split_seed in EXPECTED_SPLIT_SEEDS:
        folder = run_root / f"V2_split_{split_seed}"
        result_path = folder / "split_result.json"
        cube_path = folder / "score_cube.npz"
        assert result_path.is_file() and cube_path.is_file(), (key, split_seed, folder)
        split_result = json.loads(result_path.read_text(encoding="utf-8"))
        assert split_result["protocol_sha256"] == expected_hash["protocol_sha256"]
        assert split_result["code_sha256"] == expected_hash["code_sha256"]
        assert split_result["score_cube_sha256"] == sha256_file(cube_path)
        with np.load(cube_path, allow_pickle=False) as npz:
            base = np.asarray(npz["base"], dtype=float)
            plus = np.asarray(npz["plus"], dtype=float)
            labels = np.asarray(npz["labels"], dtype=np.int64)
            test_ids = np.asarray(npz["test_ids"], dtype=np.int64)
        assert base.shape == plus.shape == (5, 553)
        assert labels.shape == test_ids.shape == (553,)
        assert len(np.unique(test_ids)) == 553
        base_mean = mean_macro(base, labels)
        plus_mean = mean_macro(plus, labels)
        assert np.isclose(base_mean, split_result["base_mean_macro_f1"], atol=1e-12)
        assert np.isclose(plus_mean, split_result["plus_mean_macro_f1"], atol=1e-12)
        final_row = final_rows[split_seed]
        assert np.isclose(base_mean, final_row["base"], atol=1e-12)
        assert np.isclose(plus_mean, final_row["plus"], atol=1e-12)
        order = np.argsort(test_ids)
        verified[key][split_seed] = {
            "test_ids_sorted": test_ids[order],
            "labels_sorted": labels[order],
            "base_mean": base_mean,
            "plus_mean": plus_mean,
            "score_cube_sha256": split_result["score_cube_sha256"],
        }

# H1: six matched within-model comparisons, Holm family of six.
h1_rows = []
for key, labels in EXPECTED.items():
    path, item = by_key[key]
    block = item["v2_20split"]
    recomputed_deltas = [
        verified[key][seed]["plus_mean"] - verified[key][seed]["base_mean"]
        for seed in EXPECTED_SPLIT_SEEDS
    ]
    stats = nb_stats(recomputed_deltas)
    nb = block["nadeau_bengio"]
    assert np.isclose(stats["mean_delta"], block["mean_delta"], atol=1e-12)
    assert np.allclose(stats["ci95"], nb["ci95"], atol=1e-12)
    assert np.isclose(stats["p_two_sided_vs_zero"], nb["p_two_sided_raw"], atol=1e-12)
    h1_rows.append({
        "model_key": key,
        "baseline": labels[0],
        "plus": labels[1],
        "baseline_macro_f1": float(np.mean([verified[key][s]["base_mean"] for s in EXPECTED_SPLIT_SEEDS])),
        "plus_macro_f1": float(np.mean([verified[key][s]["plus_mean"] for s in EXPECTED_SPLIT_SEEDS])),
        "delta": stats["mean_delta"],
        "ci95_lower": stats["ci95"][0],
        "ci95_upper": stats["ci95"][1],
        "p_raw": stats["p_two_sided_vs_zero"],
        "splits_plus_better": int(np.sum(np.asarray(recomputed_deltas) > 0)),
        "source_final_summary": str(path),
    })

for row, p_holm in zip(h1_rows, holm_adjust([row["p_raw"] for row in h1_rows])):
    row["p_holm"] = float(p_holm)
    row["meets_effect_size"] = bool(row["delta"] >= MINIMUM_PRACTICAL_DELTA)
    row["nb_ci_excludes_zero"] = bool(row["ci95_lower"] > 0)
    row["holm_rejects"] = bool(p_holm < ALPHA)
    row["meets_predeclared_rule"] = bool(
        row["meets_effect_size"] and row["nb_ci_excludes_zero"] and row["holm_rejects"]
    )

# H2: RF+/LightGBM+ versus reproduced original Meta-IFD baseline.
# Pairing is allowed only after exact test-ID and label equality is verified.
h2_specs = [("H2a", "random_forest", "RF+"), ("H2b", "lightgbm", "LightGBM+")]
h2_rows = []
for hypothesis, tree_key, tree_label in h2_specs:
    deltas = []
    for split_seed in EXPECTED_SPLIT_SEEDS:
        tree = verified[tree_key][split_seed]
        meta = verified["metaifd"][split_seed]
        assert np.array_equal(tree["test_ids_sorted"], meta["test_ids_sorted"]), (
            hypothesis, split_seed, "test IDs"
        )
        assert np.array_equal(tree["labels_sorted"], meta["labels_sorted"]), (
            hypothesis, split_seed, "labels"
        )
        deltas.append(tree["plus_mean"] - meta["base_mean"])
    stats = nb_stats(deltas)
    h2_rows.append({
        "hypothesis": hypothesis,
        "candidate": tree_label,
        "comparator": "Meta-IFD",
        "candidate_mean_macro_f1": float(np.mean([
            verified[tree_key][s]["plus_mean"] for s in EXPECTED_SPLIT_SEEDS
        ])),
        "comparator_mean_macro_f1": float(np.mean([
            verified["metaifd"][s]["base_mean"] for s in EXPECTED_SPLIT_SEEDS
        ])),
        **stats,
        "test_ids_and_labels_matched_all_splits": True,
        "noninferiority_margin": NONINFERIORITY_MARGIN,
    })

ni_holm = holm_adjust([row["p_one_sided_noninferiority_raw"] for row in h2_rows])
sup_holm = holm_adjust([row["p_one_sided_superiority_raw"] for row in h2_rows])
for row, p_ni_holm, p_sup_holm in zip(h2_rows, ni_holm, sup_holm):
    row["p_noninferiority_holm"] = float(p_ni_holm)
    row["p_superiority_holm"] = float(p_sup_holm)
    row["noninferior"] = bool(
        row["ci95"][0] > NONINFERIORITY_MARGIN and p_ni_holm < H2_ONE_SIDED_ALPHA
    )
    row["superior_if_gated"] = bool(
        row["noninferior"] and row["ci95"][0] > 0 and p_sup_holm < H2_ONE_SIDED_ALPHA
    )

output = Path("/kaggle/working/matched_v2_1_1_20split_final_aggregation")
output.mkdir(parents=True, exist_ok=False)
for filename, rows in (
    ("matched_v2_1_1_h1_6pairs_holm.csv", h1_rows),
    ("matched_v2_1_1_h2_noninferiority.csv", h2_rows),
):
    with (output / filename).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

summary = {
    "status": "complete v2.1.1 aggregation with full score-cube verification",
    "master_protocol_sha256": MASTER_PROTOCOL_SHA256,
    "expected_hashes": EXPECTED_HASHES,
    "all_six_pairs_present": True,
    "all_twenty_splits_present_per_pair": True,
    "all_score_cubes_recomputed": True,
    "h1_within_pair_holm_family_size": 6,
    "h1_rows": h1_rows,
    "h2_primary_comparator": "reproduced original Meta-IFD baseline",
    "h2_noninferiority_margin": NONINFERIORITY_MARGIN,
    "h2_one_sided_alpha": H2_ONE_SIDED_ALPHA,
    "h2_holm_family_size": 2,
    "h2_rows": h2_rows,
    "h3_metaifd_plus": "not confirmatory; omitted from inferential conclusions",
    "cross_session_caveat": (
        "test IDs and labels matched exactly; separately materialized neighborhood/UFE "
        "samples may differ across sessions"
    ),
}
(output / "matched_v2_1_1_final_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)

lines = [
    "# matched v2.1.1 — verified 20-split results",
    "",
    "## H1: within-model UFE effect (Holm across six pairs)",
    "",
    "| Pair | Baseline | Plus | Delta pp | NB 95% CI pp | p Holm | Rule |",
    "|---|---:|---:|---:|---:|---:|:---:|",
]
for row in h1_rows:
    lines.append(
        f"| {row['baseline']} vs {row['plus']} | {row['baseline_macro_f1']*100:.2f} | "
        f"{row['plus_macro_f1']*100:.2f} | {row['delta']*100:+.2f} | "
        f"[{row['ci95_lower']*100:+.2f}, {row['ci95_upper']*100:+.2f}] | "
        f"{row['p_holm']:.6g} | {'PASS' if row['meets_predeclared_rule'] else 'FAIL'} |"
    )
lines += [
    "",
    "## H2: RF+/LightGBM+ versus original Meta-IFD",
    "",
    "| Hypothesis | Candidate | Delta pp | NB 95% CI pp | p NI Holm | Non-inferior | Superior |",
    "|---|---|---:|---:|---:|:---:|:---:|",
]
for row in h2_rows:
    lines.append(
        f"| {row['hypothesis']} | {row['candidate']} | {row['mean_delta']*100:+.2f} | "
        f"[{row['ci95'][0]*100:+.2f}, {row['ci95'][1]*100:+.2f}] | "
        f"{row['p_noninferiority_holm']:.6g} | "
        f"{'YES' if row['noninferior'] else 'NO'} | "
        f"{'YES' if row['superior_if_gated'] else 'NO'} |"
    )
(output / "matched_v2_1_1_final_tables.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
print("Output:", output)
